## Analyse synthetic combinations (ARCHIVED — not used by any manuscript figure or table)

Kept for reference: forest plots of 'incomplete' LR/GM pairings across cohorts. Paths point at the final layout; outputs were cleared.

This notebook is a **portable copy** of the ideas from `4.2_analyse_synthetic_combinations.ipynb`, adapted to the paths/data layout of this `framework_repository`.

### Goal
- **Identify what “types” (LR / GM) exist** for each *(feature1, feature2)* pair.
- Summarize **incomplete pairings** (pairs missing one or more types, optionally thresholded by selection %).
- Conclude with:
  - `incomplete_pairings_df`
  - One **exemplary pairing plot** (the 4-panel visualization) that you can easily adapt for other variables.

### Inputs
- Synthetic feature selection tables:
  - `results_taxa/synthetic_features_taxa.csv`
  - `results_functions/synthetic_features_functions.csv`
  - `results_combined/synthetic_features_combined.csv`
- Cohort-level matrices (for plotting):
  - `prepared_data/X_*_{data_type}.tsv`
  - `prepared_data/y_*_{data_type}.tsv` (falls back to any `y_*` for the cohort if a matching `{data_type}` file is missing)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Synthetic feature types to consider.
# NOTE: MULT_* combinations are intentionally excluded for now.
SYNTHETIC_TYPES = ("LR", "GM")

# --- knobs you will most likely tweak ---
MIN_SELECTION_THRESHOLD = 50.0  # keep incomplete pairs where max(Selection %) >= this
PREFERRED_COHORTS = [
    "Public_study__YachidaS_2019",
    "Public_study__ZellerG_2014",
    "Public_study__WirbelJ_2018",
    "Public_study__VogtmannE_2016",
]

# If you want to force a specific example later, set these (else we auto-pick from incomplete_pairings_df)
EXAMPLE_DATA_TYPE = None   # "taxa" | "functions" | "combined" | None
EXAMPLE_FEATURE1 = None
EXAMPLE_FEATURE2 = None
EXAMPLE_FEATURE_TYPE = None  # "LR" | "GM" | None

RESULTS_FILES = {
    "taxa": "../M2_loco_crc_pipeline/results/taxa/synthetic_features_taxa.csv",
    "functions": "../M2_loco_crc_pipeline/results/functions/synthetic_features_functions.csv",
    "combined": "../M2_loco_crc_pipeline/results/combined/synthetic_features_combined.csv",
}


In [ ]:
def parse_synthetic_feature(feature_name: str):
    """Parse: {TYPE}_{feature1}__{feature2} where TYPE ∈ {LR, GM}.

Returns (feature_type, feature1, feature2) and sorts (feature1, feature2) alphabetically
to make pair-grouping stable.

NOTE: MULT_* combinations are intentionally ignored for now.
"""
    if not isinstance(feature_name, str):
        return None, None, None

    for prefix in ("LR_", "GM_"):
        if feature_name.startswith(prefix):
            feature_type = prefix[:-1]
            remaining = feature_name[len(prefix):]
            break
    else:
        return None, None, None

    if "__" not in remaining:
        return feature_type, remaining, None

    f1, f2 = remaining.split("__", 1)
    if f1 > f2:
        f1, f2 = f2, f1
    return feature_type, f1, f2


def load_synthetic_features(data_type: str) -> pd.DataFrame:
    path = RESULTS_FILES[data_type]
    df = pd.read_csv(path)

    required = {"Feature", "Selection Count", "Selection %"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {sorted(missing)}")

    parsed = df["Feature"].apply(parse_synthetic_feature)
    df[["Feature_Type", "Feature1", "Feature2"]] = pd.DataFrame(parsed.tolist(), index=df.index)
    df["Data_Type"] = data_type

    # Drop any malformed rows
    df = df.dropna(subset=["Feature_Type", "Feature1", "Feature2"]).copy()
    return df


def build_pair_type_table(parsed_df: pd.DataFrame) -> pd.DataFrame:
    all_types = set(SYNTHETIC_TYPES)
    expected_n_types = len(all_types)

    grouped = (
        parsed_df.groupby(["Data_Type", "Feature1", "Feature2"], as_index=False)
        .agg(
            Types_Present=("Feature_Type", lambda s: sorted(set(s))),
            Num_Types=("Feature_Type", lambda s: len(set(s))),
            Max_Selection_Pct=("Selection %", "max"),
        )
    )

    grouped["Missing_Types"] = grouped["Types_Present"].apply(lambda xs: sorted(all_types - set(xs)))

    def _category(row):
        present = row["Types_Present"]
        missing = row["Missing_Types"]
        if row["Num_Types"] == expected_n_types:
            return f"complete ({'+'.join(SYNTHETIC_TYPES)})"
        if row["Num_Types"] == expected_n_types - 1:
            return f"missing {','.join(missing)}"
        if row["Num_Types"] == 1:
            return f"only {present[0]}"
        return "other"

    grouped["Category"] = grouped.apply(_category, axis=1)
    return grouped


def filter_incomplete_pairs(pair_df: pd.DataFrame, min_selection_threshold: float) -> pd.DataFrame:
    incomplete = pair_df[(pair_df["Num_Types"] < len(SYNTHETIC_TYPES)) & (pair_df["Max_Selection_Pct"] >= min_selection_threshold)].copy()
    incomplete = incomplete.sort_values(["Num_Types", "Max_Selection_Pct"], ascending=[True, False])
    return incomplete


# --- load + summarize ---
parsed_by_type = {dt: load_synthetic_features(dt) for dt in RESULTS_FILES}
pair_tables = {dt: build_pair_type_table(parsed_by_type[dt]) for dt in RESULTS_FILES}

for dt, pair_df in pair_tables.items():
    print("=" * 80)
    print(f"PAIR TYPE SUMMARY: {dt}")
    print("=" * 80)
    print(pair_df["Category"].value_counts().to_string())


# --- incomplete pairings (main output df) ---
incomplete_pairings_df = pd.concat(
    [filter_incomplete_pairs(pair_tables[dt], MIN_SELECTION_THRESHOLD) for dt in RESULTS_FILES],
    ignore_index=True,
)

# --- final detailed tables (by pairing type + data type) ---
# Add per-type Selection% columns (LR/GM) for each (Feature1, Feature2) pair.
from IPython.display import display


def build_pair_type_detail_table(parsed_df: pd.DataFrame, pair_df: pd.DataFrame) -> pd.DataFrame:
    pivot_sel = (
        parsed_df.pivot_table(
            index=["Data_Type", "Feature1", "Feature2"],
            columns="Feature_Type",
            values="Selection %",
            aggfunc="max",
        )
        .reset_index()
    )

    # Normalize expected columns (some pairs won't have all types)
    for col in SYNTHETIC_TYPES:
        if col not in pivot_sel.columns:
            pivot_sel[col] = np.nan

    pivot_sel = pivot_sel.rename(
        columns={
            "LR": "LR_Selection%",
            "GM": "GM_Selection%",
        }
    )

    out = pair_df.merge(pivot_sel, on=["Data_Type", "Feature1", "Feature2"], how="left")

    # Helpful ordering for review
    out = out.sort_values(["Data_Type", "Num_Types", "Max_Selection_Pct"], ascending=[True, True, False])
    return out


final_tables_by_data_type = {dt: build_pair_type_detail_table(parsed_by_type[dt], pair_tables[dt]) for dt in RESULTS_FILES}

# Apply the prevalence (selection %) threshold to the FINAL tables for all data types
# (same logic as used for taxa; now enforced for functions + combined too).
final_tables_by_data_type = {
    dt: df[(df["Num_Types"] < len(SYNTHETIC_TYPES)) & (df["Max_Selection_Pct"] >= MIN_SELECTION_THRESHOLD)].copy()
    for dt, df in final_tables_by_data_type.items()
}

final_pair_type_table = pd.concat(final_tables_by_data_type.values(), ignore_index=True)

print("\n" + "=" * 80)
print(f"FINAL TABLES (INCOMPLETE PAIRS, Max Selection % ≥ {MIN_SELECTION_THRESHOLD})")
print("=" * 80)

for dt in ["taxa", "functions", "combined"]:
    df_dt = final_tables_by_data_type[dt]

    print("\n" + "-" * 80)
    print(f"DATA TYPE: {dt}")
    print("-" * 80)
    print(f"Incomplete pairs (thresholded): {len(df_dt)}")

    if len(df_dt) > 0:
        print("\nCategory counts:")
        print(df_dt["Category"].value_counts().to_string())

    display(df_dt)

# Show the df as the final output of this cell
incomplete_pairings_df


In [ ]:
def load_X_y(cohort_stem: str, data_type: str):
    """Load X and y from prepared_data.

Expected:
- X: prepared_data/X_{cohort_stem}_{data_type}.tsv (index: sample IDs)
- y: prepared_data/y_{cohort_stem}_{data_type}.tsv (column: Study condition, values 0/1)

Convention used here (consistent with earlier notebooks):
- y == 0 → CRC
- y == 1 → Control

NOTE: Some cohorts only have a single y file (regardless of data_type). If the exact
      y_{cohort_stem}_{data_type}.tsv file is missing, we fall back to any
      y_{cohort_stem}_*.tsv that exists.
"""
    X_path = Path("../M1_data_preparation/prepared_data") / f"X_{cohort_stem}_{data_type}.tsv"
    y_path = Path("../M1_data_preparation/prepared_data") / f"y_{cohort_stem}_{data_type}.tsv"

    if not X_path.exists():
        raise FileNotFoundError(f"Missing X file: {X_path}")

    if not y_path.exists():
        candidates = sorted(Path("../M1_data_preparation/prepared_data").glob(f"y_{cohort_stem}_*.tsv"))
        if not candidates:
            raise FileNotFoundError(f"Missing y file: {y_path} (and no fallback y_{cohort_stem}_*.tsv found)")
        y_path = candidates[0]

    X = pd.read_csv(X_path, sep="\t", index_col=0)
    y_df = pd.read_csv(y_path, sep="\t", index_col=0)

    if "Study condition" not in y_df.columns:
        raise ValueError(f"Expected 'Study condition' column in {y_path}")

    y = y_df["Study condition"].astype(int)

    # Align indexes just in case
    common = X.index.intersection(y.index)
    X = X.loc[common]
    y = y.loc[common]
    return X, y


def compute_synthetic_feature(X: pd.DataFrame, feature1: str, feature2: str, feature_type: str, epsilon: float = 1e-9):
    if feature1 not in X.columns or feature2 not in X.columns:
        return None

    if feature_type not in SYNTHETIC_TYPES:
        raise ValueError(f"Unsupported feature_type={feature_type!r}. Supported: {SYNTHETIC_TYPES}")

    f1 = X[feature1].to_numpy(dtype=float)
    f2 = X[feature2].to_numpy(dtype=float)

    if feature_type == "LR":
        return np.log((f1 + epsilon) / (f2 + epsilon))
    if feature_type == "GM":
        return np.sqrt((f1 + epsilon) * (f2 + epsilon))
    return None


def plot_synthetic_feature_comparison(X, y, feature1, feature2, feature_type, cohort_name=""):
    """4-panel visualization: violin, box+points, scatter, histogram overlay."""
    synth = compute_synthetic_feature(X, feature1, feature2, feature_type)
    if synth is None:
        raise ValueError(f"Cannot compute {feature_type} for {feature1} × {feature2}")

    disease_mask = (y == 0)  # CRC
    health_mask = (y == 1)   # Control

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        f"{feature_type}: {feature1[:60]} × {feature2[:60]}\n{cohort_name}",
        fontsize=14,
        fontweight="bold",
    )

    # 1) Violin
    ax = axes[0, 0]
    data_to_plot = [synth[health_mask.to_numpy()], synth[disease_mask.to_numpy()]]
    parts = ax.violinplot(data_to_plot, positions=[0, 1], showmeans=True, showmedians=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Control", "CRC"])
    ax.set_ylabel(f"{feature_type} value")
    ax.set_title("Distribution (violin)")
    ax.grid(True, alpha=0.3)
    for pc in parts["bodies"]:
        pc.set_facecolor("lightblue")
        pc.set_alpha(0.7)

    # 2) Box + points
    ax = axes[0, 1]
    bp = ax.boxplot(data_to_plot, labels=["Control", "CRC"], patch_artist=True)
    bp["boxes"][0].set_facecolor("lightgreen")
    bp["boxes"][1].set_facecolor("lightcoral")
    for i, (mask, color) in enumerate([(health_mask, "green"), (disease_mask, "red")]):
        vals = synth[mask.to_numpy()]
        x = np.random.normal(i + 1, 0.04, size=len(vals))
        ax.scatter(x, vals, alpha=0.3, s=20, color=color)
    ax.set_ylabel(f"{feature_type} value")
    ax.set_title("Box + samples")
    ax.grid(True, alpha=0.3)

    # 3) Scatter: feature1 vs feature2
    ax = axes[1, 0]
    ax.scatter(X.loc[health_mask, feature1], X.loc[health_mask, feature2], alpha=0.6, label="Control", color="green", s=50)
    ax.scatter(X.loc[disease_mask, feature1], X.loc[disease_mask, feature2], alpha=0.6, label="CRC", color="red", s=50)
    ax.set_xlabel(feature1)
    ax.set_ylabel(feature2)
    ax.set_title("Feature1 vs Feature2")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 4) Histogram overlay
    ax = axes[1, 1]
    ax.hist(synth[health_mask.to_numpy()], bins=30, alpha=0.6, label="Control", color="green", density=True)
    ax.hist(synth[disease_mask.to_numpy()], bins=30, alpha=0.6, label="CRC", color="red", density=True)
    ax.axvline(np.mean(synth[health_mask.to_numpy()]), color="green", linestyle="--", linewidth=2)
    ax.axvline(np.mean(synth[disease_mask.to_numpy()]), color="red", linestyle="--", linewidth=2)
    ax.set_xlabel(f"{feature_type} value")
    ax.set_ylabel("Density")
    ax.set_title("Distribution overlay")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def find_cohort_with_features(data_type: str, feature1: str, feature2: str, preferred=None) -> str:
    """Find the first cohort (prepared_data) that contains both features."""
    paths = list(Path("../M1_data_preparation/prepared_data").glob(f"X_*_{data_type}.tsv"))

    def cohort_from_path(p: Path) -> str:
        suffix = f"_{data_type}.tsv"
        name = p.name
        return name[len("X_") : -len(suffix)]

    # Reorder paths to try preferred cohorts first
    if preferred:
        preferred_set = set(preferred)
        preferred_paths = [p for p in paths if cohort_from_path(p) in preferred_set]
        other_paths = [p for p in paths if cohort_from_path(p) not in preferred_set]
        paths = preferred_paths + other_paths

    for p in paths:
        cols = pd.read_csv(p, sep="\t", nrows=0, index_col=0).columns
        if feature1 in cols and feature2 in cols:
            return cohort_from_path(p)

    raise RuntimeError(f"No cohort found containing both features for data_type={data_type}")


# --- choose an example pairing ---
if EXAMPLE_DATA_TYPE and EXAMPLE_FEATURE1 and EXAMPLE_FEATURE2:
    example_data_type = EXAMPLE_DATA_TYPE
    example_f1 = EXAMPLE_FEATURE1
    example_f2 = EXAMPLE_FEATURE2
    example_type = EXAMPLE_FEATURE_TYPE or "LR"
else:
    if len(incomplete_pairings_df) == 0:
        # Fallback: pick the strongest pair from functions
        fallback = pair_tables["functions"].sort_values("Max_Selection_Pct", ascending=False).iloc[0]
        example_data_type = "functions"
        example_f1 = fallback["Feature1"]
        example_f2 = fallback["Feature2"]
        example_type = "LR"
        print("No incomplete pairs found at this threshold; using a high-selection fallback pair.")
    else:
        row = incomplete_pairings_df.iloc[0]
        example_data_type = row["Data_Type"]
        example_f1 = row["Feature1"]
        example_f2 = row["Feature2"]
        # Prefer LR if present, else the first present type
        present = row["Types_Present"]
        example_type = "LR" if "LR" in present else present[0]

if example_type not in SYNTHETIC_TYPES:
    raise ValueError(f"Example feature type {example_type!r} not in supported {SYNTHETIC_TYPES}")

cohort = find_cohort_with_features(example_data_type, example_f1, example_f2, preferred=PREFERRED_COHORTS)
X, y = load_X_y(cohort, example_data_type)

print(f"Example: data_type={example_data_type} | cohort={cohort}")
print(f"Pair: {example_f1} × {example_f2}")
print(f"Plotting type: {example_type}")

plot_synthetic_feature_comparison(X, y, example_f1, example_f2, example_type, cohort_name=cohort)


In [ ]:
# Show ALL values without truncation in notebook output
with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(incomplete_pairings_df)

# Copy a tab-delimited table to clipboard for direct paste into Excel
# (if clipboard backend is unavailable, save a TSV as fallback)
try:
    incomplete_pairings_df.to_clipboard(index=False, excel=True)
    print("Copied full table to clipboard. Paste directly into Excel (Ctrl/Cmd+V).")
except Exception as e:
    out_tsv = "incomplete_pairings_df_full.tsv"
    incomplete_pairings_df.to_csv(out_tsv, sep="\t", index=False)
    print(f"Clipboard copy failed ({e}). Saved full table to {out_tsv} instead.")

In [ ]:
# --- Validation: forest plots across cohorts for each incomplete pairing ---
# For each row in `incomplete_pairings_df`, compute cohort-level effect sizes (CRC vs Control)
# and show them as a forest-style plot.
#
# Effect size: rank-biserial correlation (rbc) derived from Mann–Whitney U
#   rbc = 2*U/(n_crc*n_ctrl) - 1
#   positive rbc => higher values in CRC than Control
#
# Significance: Mann–Whitney U p-values (two-sided) per cohort, plus BH-FDR correction across cohorts.

from pathlib import Path


def _bh_fdr(pvals):
    """Benjamini–Hochberg FDR. Returns q-values aligned to pvals."""
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q

    p_ok = p[ok]
    m = p_ok.size
    order = np.argsort(p_ok)
    ranked = p_ok[order]

    q_ranked = ranked * m / (np.arange(1, m + 1))
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q_ok = np.empty_like(p_ok)
    q_ok[order] = q_ranked
    q[ok] = q_ok
    return q


def _cohort_stems_for_data_type(data_type: str):
    paths = sorted(Path("../M1_data_preparation/prepared_data").glob(f"X_*_{data_type}.tsv"))
    suffix = f"_{data_type}.tsv"
    out = []
    for p in paths:
        name = p.name
        if not name.startswith("X_") or not name.endswith(suffix):
            continue
        out.append(name[len("X_") : -len(suffix)])
    return out


def _compute_rbc_and_p(disease_vals: np.ndarray, health_vals: np.ndarray):
    """Returns (rbc, p). Drops non-finite values before testing."""
    from scipy.stats import mannwhitneyu

    disease_vals = np.asarray(disease_vals, dtype=float)
    health_vals = np.asarray(health_vals, dtype=float)

    disease_vals = disease_vals[np.isfinite(disease_vals)]
    health_vals = health_vals[np.isfinite(health_vals)]

    n_crc = len(disease_vals)
    n_ctrl = len(health_vals)
    if n_crc == 0 or n_ctrl == 0:
        return np.nan, np.nan

    try:
        U, p = mannwhitneyu(disease_vals, health_vals, alternative="two-sided")
    except Exception:
        return np.nan, 1.0

    denom = n_crc * n_ctrl
    rbc = (2 * U / denom - 1) if denom > 0 else np.nan
    return float(rbc), float(p)


def _bootstrap_rbc_ci(disease_vals: np.ndarray, health_vals: np.ndarray, n_boot: int = 200, seed: int = 0):
    """Bootstrap CI for rbc. Returns (lo, hi). Drops non-finite first."""
    disease_vals = np.asarray(disease_vals, dtype=float)
    health_vals = np.asarray(health_vals, dtype=float)
    disease_vals = disease_vals[np.isfinite(disease_vals)]
    health_vals = health_vals[np.isfinite(health_vals)]

    if len(disease_vals) < 3 or len(health_vals) < 3 or n_boot <= 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        d = rng.choice(disease_vals, size=len(disease_vals), replace=True)
        h = rng.choice(health_vals, size=len(health_vals), replace=True)
        rbc, _p = _compute_rbc_and_p(d, h)
        if np.isfinite(rbc):
            boots.append(rbc)

    if len(boots) < max(20, int(n_boot * 0.2)):
        return np.nan, np.nan

    lo, hi = np.percentile(np.asarray(boots), [2.5, 97.5])
    return float(lo), float(hi)


def compute_pair_validation_table(data_type: str, feature1: str, feature2: str, feature_type: str, n_boot: int = 200):
    """Compute per-cohort rbc + p + BH-q for one (feature1, feature2, type) in one data_type."""
    cohorts = _cohort_stems_for_data_type(data_type)
    rows = []

    for cohort in cohorts:
        try:
            X, y = load_X_y(cohort, data_type)
        except Exception:
            continue

        synth = compute_synthetic_feature(X, feature1, feature2, feature_type)
        if synth is None:
            continue

        disease_mask = (y == 0)
        health_mask = (y == 1)

        dvals = np.asarray(synth, dtype=float)[disease_mask.to_numpy()]
        hvals = np.asarray(synth, dtype=float)[health_mask.to_numpy()]

        # Drop non-finite values and SKIP cohorts with empty groups (prevents NaN p-values)
        dvals = dvals[np.isfinite(dvals)]
        hvals = hvals[np.isfinite(hvals)]
        if len(dvals) == 0 or len(hvals) == 0:
            continue

        rbc, p = _compute_rbc_and_p(dvals, hvals)
        lo, hi = _bootstrap_rbc_ci(
            dvals,
            hvals,
            n_boot=n_boot,
            seed=hash((data_type, cohort, feature1, feature2, feature_type)) % (2**32),
        )

        rows.append(
            {
                "data_type": data_type,
                "feature_type": feature_type,
                "cohort": cohort,
                "n_crc": int(disease_mask.sum()),
                "n_ctrl": int(health_mask.sum()),
                "rbc": rbc,
                "rbc_ci_lo": lo,
                "rbc_ci_hi": hi,
                "p": p,
            }
        )

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    df["q_fdr"] = _bh_fdr(df["p"].values)

    df["cohort_short"] = (
        df["cohort"]
        .str.replace("Public_study__", "", regex=False)
        .str.replace("This_study_", "", regex=False)
    )

    df["abs_rbc"] = df["rbc"].abs()
    df = df.sort_values(["q_fdr", "abs_rbc"], ascending=[True, False]).reset_index(drop=True)
    return df


def plot_forest_for_pair(row, n_boot: int = 200, show_both_types: bool = True):
    """Forest-style plot for one row from incomplete_pairings_df."""
    data_type = row["Data_Type"]
    f1 = row["Feature1"]
    f2 = row["Feature2"]

    present_types = row.get("Types_Present", [])
    if isinstance(present_types, str):
        present_types = [present_types]

    types_to_plot = []
    if show_both_types:
        for t in ["LR", "GM"]:
            if t in present_types:
                types_to_plot.append(t)
    if not types_to_plot:
        types_to_plot = ["LR", "GM"]

    tables = []
    for t in types_to_plot:
        df = compute_pair_validation_table(data_type, f1, f2, t, n_boot=n_boot)
        if len(df) > 0:
            tables.append((t, df))

    if not tables:
        print(f"⚠ No cohort-level results for {data_type}: {f1} × {f2} (tried {types_to_plot})")
        return

    ncols = len(tables)
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(10 * ncols, 0.35 * max(len(tables[0][1]), 8) + 3),
        sharey=False,
    )
    if ncols == 1:
        axes = [axes]

    for ax, (t, df) in zip(axes, tables):
        y = np.arange(len(df))
        x = df["rbc"].values

        lo = df["rbc_ci_lo"].values
        hi = df["rbc_ci_hi"].values
        have_ci = np.isfinite(lo) & np.isfinite(hi)

        colors = np.where(df["q_fdr"].values < 0.05, "tab:red", "tab:blue")

        ax.scatter(x, y, c=colors, s=35, alpha=0.9, edgecolors="black", linewidth=0.3)

        if have_ci.any():
            xerr = np.vstack([x - lo, hi - x])
            xerr[:, ~have_ci] = 0
            ax.errorbar(x, y, xerr=xerr, fmt="none", ecolor="gray", elinewidth=1, capsize=2, alpha=0.8)

        ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
        ax.set_yticks(y)
        ax.set_yticklabels(df["cohort_short"].tolist(), fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Rank-biserial r (CRC vs Control)")
        ax.set_title(f"{data_type} | {t}\n{f1} × {f2}")
        ax.grid(axis="x", alpha=0.25)

        for i in range(len(df)):
            p = df.loc[i, "p"]
            q = df.loc[i, "q_fdr"]
            ax.text(
                1.02,
                i,
                f"p={p:.2e}\nq={q:.2e}",
                transform=ax.get_yaxis_transform(),
                va="center",
                fontsize=7,
            )

    plt.tight_layout()
    plt.show()


# --- run the validation plots ---
# WARNING: this can generate MANY plots if `incomplete_pairings_df` is large.
# Set MAX_PAIRS_TO_PLOT to a smaller number while iterating.
MAX_PAIRS_TO_PLOT = None  # e.g. 10
N_BOOT = 200              # increase for smoother CIs (slower)

_df = incomplete_pairings_df.copy()
if MAX_PAIRS_TO_PLOT is not None:
    _df = _df.head(MAX_PAIRS_TO_PLOT)

print(f"Will plot {len(_df)} pair(s) from incomplete_pairings_df")

for _i, _row in _df.iterrows():
    plot_forest_for_pair(_row, n_boot=N_BOOT, show_both_types=True)

In [ ]:
# --- Compact figure for main text: small-multiples (mini-forest) across cohorts ---
# This is a *condensed* visualization for a subset of pairs (e.g. top N), suitable for main text.
# It does NOT replace the detailed forest plots you may generate for the supplement.
#
# What it shows (per panel = one pair):
# - x: effect size = rank-biserial correlation (rbc) from Mann–Whitney U, CRC vs Control
# - dots: cohorts (no per-cohort p/q text to keep it compact)
# - red dot: cohort significant after BH-FDR across cohorts *within that pair* (q<0.05)
# - blue dot: not significant after FDR
# - title includes: data type, feature type used, number of cohorts used, and meta q (optional)

from pathlib import Path


def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q

    p_ok = p[ok]
    m = p_ok.size
    order = np.argsort(p_ok)
    ranked = p_ok[order]

    q_ranked = ranked * m / (np.arange(1, m + 1))
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q_ok = np.empty_like(p_ok)
    q_ok[order] = q_ranked
    q[ok] = q_ok
    return q


def _cohort_stems_for_data_type(data_type: str):
    paths = sorted(Path("../M1_data_preparation/prepared_data").glob(f"X_*_{data_type}.tsv"))
    suffix = f"_{data_type}.tsv"
    out = []
    for p in paths:
        name = p.name
        if not name.startswith("X_") or not name.endswith(suffix):
            continue
        out.append(name[len("X_") : -len(suffix)])
    return out


def _compute_rbc_and_p(disease_vals: np.ndarray, health_vals: np.ndarray):
    from scipy.stats import mannwhitneyu

    disease_vals = np.asarray(disease_vals, dtype=float)
    health_vals = np.asarray(health_vals, dtype=float)
    disease_vals = disease_vals[np.isfinite(disease_vals)]
    health_vals = health_vals[np.isfinite(health_vals)]

    n_crc = len(disease_vals)
    n_ctrl = len(health_vals)
    if n_crc == 0 or n_ctrl == 0:
        return np.nan, np.nan, n_crc, n_ctrl

    try:
        U, p = mannwhitneyu(disease_vals, health_vals, alternative="two-sided")
    except Exception:
        return np.nan, 1.0, n_crc, n_ctrl

    denom = n_crc * n_ctrl
    rbc = (2 * U / denom - 1) if denom > 0 else np.nan
    return float(rbc), float(p), n_crc, n_ctrl


def _meta_stouffer_signed(pvals: np.ndarray, signs: np.ndarray, weights: np.ndarray):
    """Signed Stouffer Z meta-analysis. Returns (z_meta, p_meta)."""
    from scipy.stats import norm

    p = np.asarray(pvals, dtype=float)
    s = np.asarray(signs, dtype=float)
    w = np.asarray(weights, dtype=float)

    ok = np.isfinite(p) & (p > 0) & (p <= 1) & np.isfinite(s) & (s != 0) & np.isfinite(w) & (w > 0)
    if ok.sum() == 0:
        return np.nan, np.nan

    z = norm.isf(p[ok] / 2.0) * np.sign(s[ok])
    w_ok = w[ok]

    z_meta = np.sum(w_ok * z) / np.sqrt(np.sum(w_ok ** 2)) if np.sum(w_ok ** 2) > 0 else np.nan
    p_meta = 2 * norm.sf(abs(z_meta)) if np.isfinite(z_meta) else np.nan
    return float(z_meta), float(p_meta)


def _pick_feature_type_for_row(row):
    """Prefer LR if present, else GM, else LR fallback."""
    present = row.get("Types_Present", [])
    if isinstance(present, str):
        present = [present]
    if "LR" in present:
        return "LR"
    if "GM" in present:
        return "GM"
    return "LR"


def compute_pair_compact_table(row):
    """Per-cohort rbc/p/q for one incomplete_pairings_df row."""
    data_type = row["Data_Type"]
    f1 = row["Feature1"]
    f2 = row["Feature2"]
    feature_type = _pick_feature_type_for_row(row)

    cohorts = _cohort_stems_for_data_type(data_type)
    rows = []

    for cohort in cohorts:
        try:
            X, y = load_X_y(cohort, data_type)
        except Exception:
            continue

        synth = compute_synthetic_feature(X, f1, f2, feature_type)
        if synth is None:
            continue

        disease_mask = (y == 0)
        health_mask = (y == 1)

        dvals = np.asarray(synth)[disease_mask.to_numpy()]
        hvals = np.asarray(synth)[health_mask.to_numpy()]

        rbc, p, n_crc, n_ctrl = _compute_rbc_and_p(dvals, hvals)
        if not np.isfinite(p) or not np.isfinite(rbc) or n_crc == 0 or n_ctrl == 0:
            continue

        rows.append(
            {
                "cohort": cohort,
                "cohort_short": str(cohort).replace("Public_study__", "").replace("This_study_", ""),
                "rbc": rbc,
                "p": p,
                "n_crc": n_crc,
                "n_ctrl": n_ctrl,
            }
        )

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return feature_type, df, (np.nan, np.nan)

    df["q"] = _bh_fdr(df["p"].values)

    # meta p/q (across cohorts) – optional but useful for a compact title
    signs = df["rbc"].values
    weights = np.sqrt(df["n_crc"].values + df["n_ctrl"].values)
    z_meta, p_meta = _meta_stouffer_signed(df["p"].values, signs, weights)

    return feature_type, df.sort_values("q", ascending=True).reset_index(drop=True), (z_meta, p_meta)


# ----------------- knobs -----------------
N_PAIRS = 6                # number of pairs to show
N_COLS = 3                 # subplot columns
FIG_DPI = 200
SHOW_META_P = True         # show meta p in title

# Choose which pairs to include (default: strongest by Max_Selection_Pct)
# You can replace this selection with a manual list if desired.
_pairs = incomplete_pairings_df.copy()
_pairs = _pairs.sort_values(["Max_Selection_Pct"], ascending=False).head(N_PAIRS).reset_index(drop=True)

# ----------------- compute all panels first (so we can share x-limits) -----------------
panels = []
all_rbc = []
for _, r in _pairs.iterrows():
    feature_type, df, meta = compute_pair_compact_table(r)
    panels.append((r, feature_type, df, meta))
    if len(df) > 0:
        all_rbc.extend(df["rbc"].values.tolist())

if len(all_rbc) == 0:
    print("⚠ No cohort-level results available for the selected pairs.")
else:
    xlim = max(0.25, float(np.nanmax(np.abs(all_rbc))))
    xlim = min(1.0, xlim)  # rbc is in [-1, 1]

    nrows = int(np.ceil(len(panels) / N_COLS))
    fig, axes = plt.subplots(nrows, N_COLS, figsize=(6.2 * N_COLS, 2.4 * nrows), dpi=FIG_DPI, sharex=True)
    axes = np.atleast_1d(axes).reshape(nrows, N_COLS)

    for i, (row, feature_type, df, (z_meta, p_meta)) in enumerate(panels):
        ax = axes[i // N_COLS, i % N_COLS]

        data_type = row["Data_Type"]
        f1 = row["Feature1"]
        f2 = row["Feature2"]

        if len(df) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue

        y = np.arange(len(df))
        colors = np.where(df["q"].values < 0.05, "tab:red", "tab:blue")
        ax.scatter(df["rbc"].values, y, c=colors, s=18, alpha=0.9, edgecolors="black", linewidth=0.25)
        ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.7)

        # Minimal y-axis: show only a few cohort labels if many
        if len(df) <= 8:
            ax.set_yticks(y)
            ax.set_yticklabels(df["cohort_short"].tolist(), fontsize=7)
        else:
            ax.set_yticks([])

        ax.invert_yaxis()
        ax.set_xlim(-xlim, xlim)
        ax.grid(axis="x", alpha=0.2)

        title = f"{data_type} | {feature_type}\n{f1} × {f2}\n(n={len(df)} cohorts)"
        if SHOW_META_P and np.isfinite(p_meta):
            title += f" | meta p={p_meta:.1e}"
        ax.set_title(title, fontsize=8, fontweight="bold")

    # Turn off any unused axes
    for j in range(len(panels), nrows * N_COLS):
        axes[j // N_COLS, j % N_COLS].set_axis_off()

    fig.supxlabel("Effect size R (CRC vs Control)")
    fig.supylabel("Cohorts (labels omitted if many)")
    fig.suptitle("Synthetic pair validation across cohorts (compact)", fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    print("Legend:")
    print("  red dot = cohort significant after BH-FDR within that pair (q<0.05)")
    print("  blue dot = not significant after BH-FDR")


In [ ]:
# --- Compact figure (all 11 original pairings) ---
# Paper-ready compiled small multiples.
# - 3 subfigures per row
# - a), b), c)... panel labels
# - larger writing
# - dashed light-grey lollipop lines indicate uncorrected p<0.05 per cohort
# - if a name looks like PWY-xxxx:..., show only PWY-xxxx in the subtitle and print the full name as caption text

N_PAIRS = 11
N_COLS = 3
FIG_DPI = 300
SHOW_META_P = True
P_UNCORR_SIG = 0.05

# Narrower panels + larger text (main-text layout)
PANEL_W = 4.35
PANEL_H = 3.05
TITLE_FS = 12
PARAM_FS = 10   # only the parameter-pair line
TICK_FS = 10
PANEL_LABEL_FS = 14
SUP_FS = 14


def _shorten_pwy(s):
    """If there is a ':' and 'PWY' appears before it, keep the whole prefix up to ':'.

    Examples shortened:
    - 'PWY-1234: some name' -> 'PWY-1234'
    - 'ABC-PWY: some name'  -> 'ABC-PWY'

    Returns (short, full_or_None).
    """
    if s is None:
        return "", None
    s = str(s)
    if ":" in s:
        prefix = s.split(":", 1)[0]
        if "PWY" in prefix:
            return prefix, s
    return s, None


def _truncate(s, max_len=40):
    if s is None:
        return ""
    s = str(s)
    return s if len(s) <= max_len else (s[: max_len - 1] + "…")


_pairs = incomplete_pairings_df.copy().reset_index(drop=True)
if len(_pairs) > 11:
    _pairs = _pairs.sort_values(["Max_Selection_Pct"], ascending=False).head(N_PAIRS).reset_index(drop=True)

panels = []
all_rbc = []
for _, r in _pairs.iterrows():
    feature_type, df, meta = compute_pair_compact_table(r)  # uses LR/GM availability; GM-only pairing will be GM
    panels.append((r, feature_type, df, meta))
    if len(df) > 0:
        all_rbc.extend(df["rbc"].values.tolist())

if len(all_rbc) == 0:
    print("⚠ No cohort-level results available for the selected pairs.")
else:
    xlim = max(0.25, float(np.nanmax(np.abs(all_rbc))))
    xlim = min(1.0, xlim)

    from matplotlib.gridspec import GridSpec

    n_panels = len(panels)
    nrows = int(np.ceil(n_panels / N_COLS))

    # Use a 2x finer grid to allow centering the last row (e.g., 2 panels in a 3-col layout)
    fig = plt.figure(figsize=(PANEL_W * N_COLS, PANEL_H * nrows), dpi=FIG_DPI)
    # More spacing to avoid title/subtitle overlap between rows
    gs = GridSpec(nrows, N_COLS * 2, figure=fig, wspace=0.45, hspace=0.95)

    caption_lines = []

    for i, (row, feature_type, df, (_z_meta, p_meta)) in enumerate(panels):
        r = i // N_COLS
        c = i % N_COLS
        n_in_row = min(N_COLS, n_panels - r * N_COLS)
        offset = (N_COLS - n_in_row)  # 0 for full rows, 1 if 2 panels, 2 if 1 panel

        start_col = offset + c * 2
        ax = fig.add_subplot(gs[r, start_col : start_col + 2])

        panel_label = f"{chr(ord('a') + i)})"

        data_type = row["Data_Type"]
        f1_raw = row["Feature1"]
        f2_raw = row["Feature2"]

        f1_short, f1_full = _shorten_pwy(f1_raw)
        f2_short, f2_full = _shorten_pwy(f2_raw)

        f1_disp = _truncate(f1_short, 44)
        f2_disp = _truncate(f2_short, 44)

        # Always include the FULL pairing in the caption (not only shortened PWY forms)
        caption_lines.append(f"{panel_label} {data_type} | {feature_type}: {f1_raw} × {f2_raw}")

        # Additionally keep explicit full-name notes when we shortened a PWY-like name
        if f1_full is not None:
            caption_lines.append(f"{panel_label} Feature1 full: {f1_full}")
        if f2_full is not None:
            caption_lines.append(f"{panel_label} Feature2 full: {f2_full}")

        if len(df) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
            ax.set_axis_off()
            continue

        y = np.arange(len(df))

        # FDR significance within-pair across cohorts (color)
        colors = np.where(df["q"].values < 0.05, "tab:red", "tab:blue")
        ax.scatter(
            df["rbc"].values,
            y,
            c=colors,
            s=32,
            alpha=0.97,
            edgecolors="black",
            linewidth=0.35,
            zorder=2,
        )

        ax.axvline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)

        # Keep the plot narrow: show y labels only if few cohorts
        if len(df) <= 6:
            ax.set_yticks(y)
            ax.set_yticklabels(df["cohort_short"].tolist(), fontsize=TICK_FS)
        else:
            ax.set_yticks([])

        ax.invert_yaxis()
        ax.set_xlim(-xlim, xlim)
        ax.grid(axis="x", alpha=0.25)
        ax.tick_params(axis="x", labelsize=TICK_FS)

        # Panel label
        ax.text(
            0.01,
            0.98,
            panel_label,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=PANEL_LABEL_FS,
            fontweight="bold",
        )

        # Title line (bigger) + parameter line (slightly smaller)
        title = f"{data_type} | {feature_type} | n={len(df)}"
        if SHOW_META_P and np.isfinite(p_meta):
            title += f" | meta p={p_meta:.1e}"

        # Push title higher, and place parameter line ABOVE the axes.
        ax.set_title(title, fontsize=TITLE_FS, fontweight="bold", y=1.18, pad=0)
        ax.text(
            0.5,
            1.06,
            f"{f1_disp} × {f2_disp}",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=PARAM_FS,
            clip_on=False,
        )

    fig.supxlabel("Effect size R (CRC vs Control)", fontsize=SUP_FS, fontweight="bold")
    fig.supylabel("Cohorts (labels omitted if many)", fontsize=SUP_FS, fontweight="bold")
    fig.suptitle("Synthetic pair validation across cohorts (compact)", fontsize=SUP_FS + 1, fontweight="bold", y=1.02)

    plt.show()

    print("Legend:")
    print("  dot color: red = cohort significant after BH-FDR within that pair (q<0.05); blue = not significant")

    if caption_lines:
        print("\nCaption (full PWY names):")
        for line in caption_lines:
            print("- " + line)


In [ ]:
# --- Compact figure (all 11 original pairings) + overlay other type (gray dots) ---
# Copy of the previous compact figure cell, but overlays the *other* synthetic type
# (LR vs GM) as small gray dots for the same cohorts.

N_PAIRS = 11
N_COLS = 3
FIG_DPI = 300
SHOW_META_P = True
P_UNCORR_SIG = 0.05

# Narrower panels + larger text (main-text layout)
PANEL_W = 4.35
PANEL_H = 3.05
TITLE_FS = 12
PARAM_FS = 10   # only the parameter-pair line
TICK_FS = 10
PANEL_LABEL_FS = 14
SUP_FS = 14

# Overlay appearance
ALT_DOT_COLOR = "0.65"   # gray
ALT_DOT_SIZE = 18
ALT_DOT_ALPHA = 0.9
ALT_Y_JITTER = 0.18


def _shorten_pwy(s):
    """If there is a ':' and 'PWY' appears before it, keep the whole prefix up to ':'.

    Examples shortened:
    - 'PWY-1234: some name' -> 'PWY-1234'
    - 'ABC-PWY: some name'  -> 'ABC-PWY'

    Returns (short, full_or_None).
    """
    if s is None:
        return "", None
    s = str(s)
    if ":" in s:
        prefix = s.split(":", 1)[0]
        if "PWY" in prefix:
            return prefix, s
    return s, None


def _truncate(s, max_len=40):
    if s is None:
        return ""
    s = str(s)
    return s if len(s) <= max_len else (s[: max_len - 1] + "…")


def _compute_compact_df_for_type(data_type: str, f1: str, f2: str, feature_type: str):
    """Per-cohort rbc/p/q for one (feature1, feature2, type). Matches compact schema."""
    cohorts = _cohort_stems_for_data_type(data_type)
    rows = []

    for cohort in cohorts:
        try:
            X, y = load_X_y(cohort, data_type)
        except Exception:
            continue

        synth = compute_synthetic_feature(X, f1, f2, feature_type)
        if synth is None:
            continue

        disease_mask = (y == 0)
        health_mask = (y == 1)

        dvals = np.asarray(synth)[disease_mask.to_numpy()]
        hvals = np.asarray(synth)[health_mask.to_numpy()]

        try:
            rbc, p, n_crc, n_ctrl = _compute_rbc_and_p(dvals, hvals)
        except Exception:
            continue

        if not np.isfinite(p) or not np.isfinite(rbc) or n_crc == 0 or n_ctrl == 0:
            continue

        rows.append(
            {
                "cohort": cohort,
                "cohort_short": str(cohort).replace("Public_study__", "").replace("This_study_", ""),
                "rbc": rbc,
                "p": p,
                "n_crc": n_crc,
                "n_ctrl": n_ctrl,
            }
        )

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    df["q"] = _bh_fdr(df["p"].values)
    return df.sort_values("q", ascending=True).reset_index(drop=True)


_pairs = incomplete_pairings_df.copy().reset_index(drop=True)
if len(_pairs) > 11:
    _pairs = _pairs.sort_values(["Max_Selection_Pct"], ascending=False).head(N_PAIRS).reset_index(drop=True)

panels = []
all_rbc = []
for _, r in _pairs.iterrows():
    feature_type, df, meta = compute_pair_compact_table(r)  # uses LR/GM availability; GM-only pairing will be GM

    alt_type = "GM" if feature_type == "LR" else "LR"
    alt_df = _compute_compact_df_for_type(r["Data_Type"], r["Feature1"], r["Feature2"], alt_type)

    panels.append((r, feature_type, df, meta, alt_type, alt_df))
    if len(df) > 0:
        all_rbc.extend(df["rbc"].values.tolist())
    if len(alt_df) > 0:
        all_rbc.extend(alt_df["rbc"].values.tolist())

if len(all_rbc) == 0:
    print("⚠ No cohort-level results available for the selected pairs.")
else:
    xlim = max(0.25, float(np.nanmax(np.abs(all_rbc))))
    xlim = min(1.0, xlim)

    from matplotlib.gridspec import GridSpec

    n_panels = len(panels)
    nrows = int(np.ceil(n_panels / N_COLS))

    # Use a 2x finer grid to allow centering the last row (e.g., 2 panels in a 3-col layout)
    fig = plt.figure(figsize=(PANEL_W * N_COLS, PANEL_H * nrows), dpi=FIG_DPI)
    gs = GridSpec(nrows, N_COLS * 2, figure=fig, wspace=0.45, hspace=0.95)

    caption_lines = []

    for i, (row, feature_type, df, (_z_meta, p_meta), alt_type, alt_df) in enumerate(panels):
        r = i // N_COLS
        c = i % N_COLS
        n_in_row = min(N_COLS, n_panels - r * N_COLS)
        offset = (N_COLS - n_in_row)  # 0 for full rows, 1 if 2 panels, 2 if 1 panel

        start_col = offset + c * 2
        ax = fig.add_subplot(gs[r, start_col : start_col + 2])

        panel_label = f"{chr(ord('a') + i)})"

        data_type = row["Data_Type"]
        f1_raw = row["Feature1"]
        f2_raw = row["Feature2"]

        f1_short, f1_full = _shorten_pwy(f1_raw)
        f2_short, f2_full = _shorten_pwy(f2_raw)

        f1_disp = _truncate(f1_short, 44)
        f2_disp = _truncate(f2_short, 44)

        caption_lines.append(f"{panel_label} {data_type} | {feature_type} (gray={alt_type}): {f1_raw} × {f2_raw}")

        if f1_full is not None:
            caption_lines.append(f"{panel_label} Feature1 full: {f1_full}")
        if f2_full is not None:
            caption_lines.append(f"{panel_label} Feature2 full: {f2_full}")

        if len(df) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
            ax.set_axis_off()
            continue

        df = df.sort_values("rbc")

        y = np.arange(len(df), dtype=float)
        y_map = dict(zip(df["cohort"].astype(str).tolist(), y.tolist()))

        # Main type: FDR significance within-pair across cohorts (color)
        colors = np.where(df["q"].values < 0.05, "tab:red", "tab:blue")
        ax.scatter(
            df["rbc"].values,
            y,
            c=colors,
            s=32,
            alpha=0.97,
            edgecolors="black",
            linewidth=0.35,
            zorder=3,
        )

        ax.axvline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)

        # Keep the plot narrow: show y labels only if few cohorts
        if len(df) <= 6:
            ax.set_yticks(y)
            ax.set_yticklabels(df["cohort_short"].tolist(), fontsize=TICK_FS)
        else:
            ax.set_yticks([])

        ax.invert_yaxis()
        ax.set_ylim((len(df) - 0.5) + (ALT_Y_JITTER + 0.2), -0.5 - 0.2)
        ax.set_xlim(-xlim, xlim)
        ax.grid(axis="x", alpha=0.25)
        ax.tick_params(axis="x", labelsize=TICK_FS)

        # Panel label: left of y-axis, aligned with subfigure caption (title) height
        ax.text(
            -0.08,
            1.18,
            panel_label,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=PANEL_LABEL_FS,
            fontweight="bold",
        )

        # Subfigure caption: line 1 = feature 1 (bold), line 2 = × feature 2 (bold), line 3 = feature_type | n=... (normal)
        ax.text(
            0.5,
            1.22,
            f1_disp,
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=TITLE_FS,
            fontweight="bold",
            clip_on=False,
        )
        ax.text(
            0.5,
            1.12,
            f"× {f2_disp}",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=TITLE_FS,
            fontweight="bold",
            clip_on=False,
        )
        subtitle = f"{feature_type} | n={len(df)}"
        if SHOW_META_P and np.isfinite(p_meta):
            subtitle += f" | meta p={p_meta:.1e}"
        ax.text(
            0.5,
            1.02,
            subtitle,
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=PARAM_FS,
            fontweight="normal",
            clip_on=False,
        )

    fig.supxlabel(
        "Effect size R (CRC vs Control)",
        fontsize=SUP_FS,
        fontweight="bold",
        y=0.02,  # closer to panels; tweak if needed
    )
    fig.supylabel(
        "Cohort [sorted by effect size]",
        fontsize=SUP_FS,
        fontweight="bold",
        x=0.03,  # closer to panels; tweak if needed
    )
    
    #fig.supxlabel("Effect size R (CRC vs Control)", fontsize=SUP_FS, fontweight="bold")
    #fig.supylabel("Cohorts", fontsize=SUP_FS, fontweight="bold")
    #fig.suptitle(
    #    "Synthetic pair validation across cohorts (compact; gray dots = other synthetic type)",
    #    fontsize=SUP_FS + 1,
    #     fontweight="bold",
    #     y=1.02,
    # )

    plt.show()

    print("Legend:")
    print("  main dot color: red = cohort significant after BH-FDR within that pair (q<0.05); blue = not significant")

    if caption_lines:
        print("\nCaption (full PWY names):")
        for line in caption_lines:
            print("- " + line)


In [ ]:
# --- Same subplots as last figure, but: x = original pairing effect size, y = other (never chosen) combination ---
# Run the previous cell first so _compute_compact_df_for_type and helpers are defined.
#
# INTERPRETATION:
#   - Ideal pattern: points spread along the x-axis (|x| large) but stay near y=0.
#     That means the chosen type (LR or GM) has strong effect sizes across cohorts,
#     while the other type does not → justifies why one type was selected and the other was not.
#   - Bad pattern: points follow the diagonal (y ≈ x). Then both types are similarly
#     predictive and there is no clear reason one was chosen over the other.
#   - Red dots = cohort significant (q<0.05) for the original type; we want these to
#     have y ≈ 0 (other type not significant).

N_PAIRS = 11
N_COLS = 3
FIG_DPI = 300
PANEL_W = 4.35
PANEL_H = 3.05
TITLE_FS = 12
PARAM_FS = 10
TICK_FS = 10
PANEL_LABEL_FS = 14
SUP_FS = 14

_pairs = incomplete_pairings_df.copy().reset_index(drop=True)
if len(_pairs) > N_PAIRS:
    _pairs = _pairs.sort_values(["Max_Selection_Pct"], ascending=False).head(N_PAIRS).reset_index(drop=True)

panels = []
for _, r in _pairs.iterrows():
    feature_type, df, meta = compute_pair_compact_table(r)
    alt_type = "GM" if feature_type == "LR" else "LR"
    alt_df = _compute_compact_df_for_type(r["Data_Type"], r["Feature1"], r["Feature2"], alt_type)
    panels.append((r, feature_type, df, meta, alt_type, alt_df))

from matplotlib.gridspec import GridSpec

n_panels = len(panels)
nrows = int(np.ceil(n_panels / N_COLS))
fig = plt.figure(figsize=(PANEL_W * N_COLS, PANEL_H * nrows), dpi=FIG_DPI)
gs = GridSpec(nrows, N_COLS * 2, figure=fig, wspace=0.45, hspace=0.95)

for i, (row, feature_type, df, (_z_meta, p_meta), alt_type, alt_df) in enumerate(panels):
    r = i // N_COLS
    c = i % N_COLS
    n_in_row = min(N_COLS, n_panels - r * N_COLS)
    offset = (N_COLS - n_in_row)
    start_col = offset + c * 2
    ax = fig.add_subplot(gs[r, start_col : start_col + 2])

    panel_label = f"{chr(ord('a') + i)})"
    data_type = row["Data_Type"]
    f1_raw = row["Feature1"]
    f2_raw = row["Feature2"]
    f1_short, f1_full = _shorten_pwy(f1_raw)
    f2_short, f2_full = _shorten_pwy(f2_raw)
    f1_disp = _truncate(f1_short, 44)
    f2_disp = _truncate(f2_short, 44)

    if len(df) == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
        ax.set_axis_off()
        continue

    # Merge main and other type by cohort so we have (rbc_original, rbc_other) per cohort
    main_sub = df[["cohort", "rbc", "q"]].copy()
    main_sub = main_sub.rename(columns={"rbc": "rbc_original", "q": "q_original"})
    alt_sub = alt_df[["cohort", "rbc"]].copy()
    alt_sub = alt_sub.rename(columns={"rbc": "rbc_other"})
    merged = main_sub.merge(alt_sub, on="cohort", how="inner")

    if len(merged) == 0:
        ax.text(0.5, 0.5, "No overlapping cohorts", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
        ax.set_axis_off()
        continue

    x_vals = merged["rbc_original"].values
    y_vals = merged["rbc_other"].values
    colors = np.where(merged["q_original"].values < 0.05, "tab:red", "tab:blue")

    lim = max(0.25, float(np.nanmax(np.abs(np.r_[x_vals, y_vals]))))
    lim = min(1.0, lim)
    # Shade region x > y (below diagonal)
    _x = np.linspace(-lim, lim, 200)
    ax.fill_between(_x, -lim, _x, alpha=0.2, color="gray", zorder=0)
    ax.scatter(x_vals, y_vals, c=colors, s=32, alpha=0.97, edgecolors="black", linewidth=0.35, zorder=3)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.axvline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.plot([-lim, lim], [-lim, lim], color="gray", linestyle=":", linewidth=1, alpha=0.8, label="y = x")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="both", labelsize=TICK_FS)

    ax.text(0.01, 0.98, panel_label, transform=ax.transAxes, ha="left", va="top", fontsize=PANEL_LABEL_FS, fontweight="bold")
    ax.set_title(f"{f1_disp} × {f2_disp}", fontsize=TITLE_FS, fontweight="bold", y=1.18, pad=0)
    if np.isfinite(p_meta):
        ax.text(0.5, 1.06, f"meta p = {p_meta:.1e}", transform=ax.transAxes, ha="center", va="bottom", fontsize=PARAM_FS, clip_on=False)

fig.supxlabel("Original pairing effect size (rank-biserial r)", fontsize=SUP_FS, fontweight="bold")
fig.supylabel("Other combination effect size (rank-biserial r)", fontsize=SUP_FS, fontweight="bold")
fig.suptitle("Original pairing vs other synthetic type (per cohort)", fontsize=SUP_FS + 1, fontweight="bold", y=1.02)
plt.show()
print("Legend: red = cohort significant (q<0.05) for original type; blue = not significant. Diagonal: y = x.")
print("Ideal: points horizontal (strong x, weak y); avoid points on the diagonal (both types similar).")

In [ ]:
# --- Same layout as above, but axes = effect size (Cohen's d) instead of rank-biserial r ---
# Axes = |Cohen's d| (effect strength): negative and positive d are same magnitude, opposite direction.
# x = |d| original pairing, y = |d| other combination. 0 = no effect; higher = stronger effect.

def _cohens_d_two_groups(vals0: np.ndarray, vals1: np.ndarray) -> float:
    """Cohen's d for two groups: (mean1 - mean0) / pooled_std. Returns np.nan if undefined."""
    n0, n1 = len(vals0), len(vals1)
    if n0 < 2 or n1 < 2:
        return np.nan
    m0, m1 = np.nanmean(vals0), np.nanmean(vals1)
    v0 = np.nanvar(vals0, ddof=1)
    v1 = np.nanvar(vals1, ddof=1)
    pooled_var = ((n0 - 1) * v0 + (n1 - 1) * v1) / (n0 + n1 - 2)
    if pooled_var <= 0:
        return np.nan
    return (m1 - m0) / np.sqrt(pooled_var)

def _compute_cohens_d_df_for_type(data_type: str, f1: str, f2: str, feature_type: str):
    """Per-cohort Cohen's d for one (feature1, feature2, type). Returns DataFrame with cohort, cohens_d."""
    cohorts = _cohort_stems_for_data_type(data_type)
    rows = []
    for cohort in cohorts:
        try:
            X, y = load_X_y(cohort, data_type)
        except Exception:
            continue
        synth = compute_synthetic_feature(X, f1, f2, feature_type)
        if synth is None:
            continue
        disease_mask = (y == 0)
        health_mask = (y == 1)
        d_vals = np.asarray(synth)[disease_mask.to_numpy()]
        h_vals = np.asarray(synth)[health_mask.to_numpy()]
        d_val = _cohens_d_two_groups(d_vals, h_vals)
        if not np.isfinite(d_val):
            continue
        rows.append({"cohort": cohort, "cohens_d": d_val})
    return pd.DataFrame(rows)

N_PAIRS = 11
N_COLS = 3
FIG_DPI = 300
PANEL_W = 4.35
PANEL_H = 3.05
TITLE_FS = 12
PARAM_FS = 10
TICK_FS = 10
PANEL_LABEL_FS = 14
SUP_FS = 14

_pairs = incomplete_pairings_df.copy().reset_index(drop=True)
if len(_pairs) > N_PAIRS:
    _pairs = _pairs.sort_values(["Max_Selection_Pct"], ascending=False).head(N_PAIRS).reset_index(drop=True)

panels = []
for _, r in _pairs.iterrows():
    feature_type, df, meta = compute_pair_compact_table(r)
    alt_type = "GM" if feature_type == "LR" else "LR"
    alt_df = _compute_compact_df_for_type(r["Data_Type"], r["Feature1"], r["Feature2"], alt_type)
    df_d_main = _compute_cohens_d_df_for_type(r["Data_Type"], r["Feature1"], r["Feature2"], feature_type)
    df_d_alt = _compute_cohens_d_df_for_type(r["Data_Type"], r["Feature1"], r["Feature2"], alt_type)
    panels.append((r, feature_type, df, meta, alt_type, alt_df, df_d_main, df_d_alt))

from matplotlib.gridspec import GridSpec

n_panels = len(panels)
nrows = int(np.ceil(n_panels / N_COLS))
fig = plt.figure(figsize=(PANEL_W * N_COLS, PANEL_H * nrows), dpi=FIG_DPI)
gs = GridSpec(nrows, N_COLS * 2, figure=fig, wspace=0.45, hspace=0.95)

for i, (row, feature_type, df, (_z_meta, p_meta), alt_type, alt_df, df_d_main, df_d_alt) in enumerate(panels):
    r = i // N_COLS
    c = i % N_COLS
    n_in_row = min(N_COLS, n_panels - r * N_COLS)
    offset = (N_COLS - n_in_row)
    start_col = offset + c * 2
    ax = fig.add_subplot(gs[r, start_col : start_col + 2])

    panel_label = f"{chr(ord('a') + i)})"
    data_type = row["Data_Type"]
    f1_raw = row["Feature1"]
    f2_raw = row["Feature2"]
    f1_short, f1_full = _shorten_pwy(f1_raw)
    f2_short, f2_full = _shorten_pwy(f2_raw)
    f1_disp = _truncate(f1_short, 44)
    f2_disp = _truncate(f2_short, 44)

    if len(df_d_main) == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
        ax.set_axis_off()
        continue

    main_d = df_d_main[["cohort", "cohens_d"]].rename(columns={"cohens_d": "d_original"})
    alt_d = df_d_alt[["cohort", "cohens_d"]].rename(columns={"cohens_d": "d_other"})
    merged = main_d.merge(alt_d, on="cohort", how="inner")
    if len(merged) == 0:
        ax.text(0.5, 0.5, "No overlapping cohorts", ha="center", va="center", transform=ax.transAxes, fontsize=TITLE_FS)
        ax.set_axis_off()
        continue

    merged = merged.merge(df[["cohort", "q"]].rename(columns={"q": "q_original"}), on="cohort", how="left")
    # Use |Cohen's d| so effect strength is comparable (negative d = same magnitude, opposite direction)
    x_vals = np.abs(merged["d_original"].values)
    y_vals = np.abs(merged["d_other"].values)
    colors = np.where(merged["q_original"].values < 0.05, "tab:red", "tab:blue")

    lim = max(0.25, float(np.nanmax(np.r_[x_vals, y_vals])))
    lim = min(2.0, lim)
    # Shade region x > y (original effect strength > other; below diagonal)
    _x = np.linspace(0, lim, 200)
    ax.fill_between(_x, 0, _x, alpha=0.2, color="gray", zorder=0)
    ax.scatter(x_vals, y_vals, c=colors, s=32, alpha=0.97, edgecolors="black", linewidth=0.35, zorder=3)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.axvline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.plot([0, lim], [0, lim], color="gray", linestyle=":", linewidth=1, alpha=0.8, label="y = x")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="both", labelsize=TICK_FS)

    ax.text(0.01, 0.98, panel_label, transform=ax.transAxes, ha="left", va="top", fontsize=PANEL_LABEL_FS, fontweight="bold")
    ax.set_title(f"{f1_disp} × {f2_disp}", fontsize=TITLE_FS, fontweight="bold", y=1.18, pad=0)
    if np.isfinite(p_meta):
        ax.text(0.5, 1.06, f"meta p = {p_meta:.1e}", transform=ax.transAxes, ha="center", va="bottom", fontsize=PARAM_FS, clip_on=False)

fig.supxlabel("Original pairing effect strength (|Cohen's d|)", fontsize=SUP_FS, fontweight="bold")
fig.supylabel("Other combination effect strength (|Cohen's d|)", fontsize=SUP_FS, fontweight="bold")
fig.suptitle("Original pairing vs other synthetic type (per cohort) — |Cohen's d|", fontsize=SUP_FS + 1, fontweight="bold", y=1.02)
plt.show()
print("Legend: red = cohort significant (q<0.05) for original type; blue = not significant.")
print("Axes = |Cohen's d| (effect strength); negative d same magnitude as positive, opposite direction.")
print("Ideal: points in shaded region (x > y), i.e. high x and low y.")